# module-composition — worked example 3: ModuleList registers, plain list does not

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `module-composition`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`nn.ModuleList` stores a list of child modules and registers every one, so they appear in `.parameters()`. A plain Python list of modules does NOT register them — those layers are invisible to the optimizer. This is why variable-depth models use ModuleList.

## Worked solution

We contrast the two storage choices on an N-layer MLP.

1. **ModuleList path.** `self.layers = nn.ModuleList([Linear(dim, dim) for _ in range(n)])` registers all `n` layers. `forward` iterates them with ReLU between (not after the last).
2. **Broken plain-list path.** A second class stores the same layers in a raw `[]`. Because a list is not a Module, none of those Linears register, so `.parameters()` is empty.
3. **Consequence.** An optimizer built over the broken model's `.parameters()` would update nothing.

The demo builds both, prints the ModuleList model's parameter count and the broken model's (zero), proving why ModuleList is required.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(2)

class DeepMLP(nn.Module):
    def __init__(self, dim, n):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n)])
    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = t.relu(x)
        return x

class BrokenList(nn.Module):
    def __init__(self, dim, n):
        super().__init__()
        self.layers = [nn.Linear(dim, dim) for _ in range(n)]  # NOT registered

good = DeepMLP(4, 3)
bad = BrokenList(4, 3)
print('ModuleList params:', sum(1 for _ in good.parameters()))   # 6 (3 layers x weight+bias)
print('plain-list params:', sum(1 for _ in bad.parameters()))    # 0
print('forward shape:', tuple(good(t.randn(5, 4)).shape))